# Workbook Details

**Student Name**: Gerard O'Rourke

**Student Number**: 24514772

*Workbook Structure*

1. **Report Generation Code** - The code to build/generate the report is provided first.
1. **Model Experiementation Code**
    - where I compared/experimented with different models is provided next.
    - I then include code that I used to generate the complaint / praise letters.  There was a few attempts using differnt models to try and generate "suitable" responses, to varying degrees of success.  There are more details comments in these sections
1. **Reflection**


*Comments*

As an aside I'd be interested if you had any references/examples of this being used in production as I'd be interested to see how **good, safe, reliable and consistent** responses could be created and what safe guards they might employ to ensure a sensible output.

# Section 1 Assignment - Transformers

# Task
You work for a large company and the company receives 8 letters regarding one of your products. These letters range from letters of complaint to letters of praise.

Using Hugging Face Transformers,
1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the:
    - positive responses and
    - negative responses.
1. Summarize the
    - sentiments towards the product and
    - provide a summary of the letters that reflect the most extreme sentiments.
1. Determine and report in each case on what the customer wants to happen by automatically questioning their respective letters.

Knowing how each customer feels, automatically generate replies to the two most extreme examples, including their name in the greeting in the response.


## Tools for the task
* Some great working examples can be found in the file **01_introduction.ipynb** from the [Natural Language Processing with Transformers GitHub Repository](https://github.com/nlp-with-transformers/notebooks).
* You will need to generate/provide the letters of complaint. You can generate the letters from the customers manually or automatically. You can use a Hugging Face text generation pipeline to do this or an online generator like [You.com](https://you.com/search?q=how+to+write+well&&tbm=youwrite&cfr=write&).
* You can use the library such as [python-docx](https://python-docx.readthedocs.io/en/latest/index.html#) to generate the report and the response letters.
* There is a variety of models on Hugging Face for the different tasks, e.g, [question-answering](https://huggingface.co/models?pipeline_tag=question-answering&sort=downloads) that allow you to try out different models for your task. It is strongly recommended that you review these and look at the model cards in each case. Also, you can test your text in the Hosted inference API on the right hand side of each model.

## Environment setup

In [3]:
#!pip install -q transformers torch sentencepiece

# get rid of widget error
#!pip install nbstripout
#!nbstripout /content/MN5162_Section1_Assignment_gorourke.ipynb
!wget -O /content/Image.png https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/Image.png

--2026-03-21 19:57:05--  https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/Image.png
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 509411 (497K) [image/png]
Saving to: ‘/content/Image.png’

/content/Image.png  100%[===================>] 497.47K  --.-KB/s    in 0.003s  

2026-03-21 19:57:05 (154 MB/s) - ‘/content/Image.png’ saved [509411/509411]



In [4]:
from transformers import pipeline
import pandas as pd
import textwrap
from pprint import pprint
import requests

In [5]:
# not needed for this workbook
from huggingface_hub import login
login()

### Customer Letters URLs

In [6]:
file_urls = ['https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt']

## Class to hold data as we process the feedback

In [7]:
from dataclasses import dataclass
from typing import Any

@dataclass
class CustomerFeedback:
    original_letter: str
    url : str
    customer_name: str = None
    normalised_letter:str = None
    summary: str = None
    sentiment: str = None
    sentiment_score: float = None
    sentimentIsPositive: bool  = False
    sentimentIsNegative: bool  = False
    user_request: Any = None
    response_letter: str = None

## Letter Pre-processing
1. Prior to Summary, Sentiment Analysis and Question Answering, just tidy up spaces and new lines.  
**Note:** Coverting to lowercase had a negative affect do don't do this prior to these tasks
1. Post Question answering - text can be converted to lower case.

In [8]:
import re

def preprocess_letter_text_stage1(text):
    working_text = text

    working_text = working_text.replace('\n', ' ').replace('\t', ' ')

    working_text = re.sub(r'\s+', ' ', working_text)

    return working_text.strip()

def preprocess_letter_text_stage2(text):

    working_text = text.lower()

    return working_text

## Build Models

### Build Summarisation Model
**Note**
In the example workbook, I started work before the "fixes" were available to ge the example workbook working, so I looked for another model to do this task.

See Model compare section for further comments.

**Model Selected**

facebook/bart-large-cnn

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load BART for summarization
summary_model_name = "facebook/bart-large-cnn"
summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(summary_model_name)


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [10]:
def summarise_text(model, tokenizer, text):

    # Generate summary
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=150,      # Use max_new_tokens, not max_length
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    #print("Summary:", summary)

    return summary

### Build Sentiment Model
**Model Used**

distilbert/distilbert-base-uncased-finetuned-sst-2-english

In [11]:
from transformers import pipeline

DEFAULT_SENTIMENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

classifier_pipeline = pipeline("text-classification", model = DEFAULT_SENTIMENT_MODEL)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [13]:
#
#   Naively accept the label - we see from the test data, the label is POSITIVE or NEGATIVE, and the scoring seems accurate.  Use this to return a boolean also
#
#   [{'label': 'NEGATIVE', 'score': 0.9995133876800537}]
#	[{'label': 'POSITIVE', 'score': 0.9998445510864258}]
#
def determine_sentiment(classifier, text):
    outputs = classifier(text)

    isPositive = False;
    isNegative = False;

    #
    #   should check scores too for completeness, and ensure there's only one sentiment record
    #
    for output in outputs:
        if output['label'] == "NEGATIVE":
            isNegative = True;

        if output['label'] == "POSITIVE":
            isPositive = True;

        sentiment_score = output['score']

    # return output and derived flags to simplify code later
    return outputs, isPositive, isNegative,sentiment_score

### Build Question Answering

In [14]:
DEFAULT_QA_MODEL = "distilbert/distilbert-base-cased-distilled-squad"

qa_pipeline = pipeline("question-answering", model = DEFAULT_QA_MODEL)


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
def determine_user_request(model, context) :

    question = "What does the customer want?"
    #question = "How should we respond to this letter"

    outputs = model(question=question, context=context)
    #pd.DataFrame([outputs])

    return outputs

### NER: Customer name extraction
**Note**
In the example workbook, I started work before the "fixes" were available to ge the example workbook working, so I looked for another model to do this task.

- The default model, extracted the Customer name successfully so decided to proceed with this, **once** the **aggregation_strategy** was changed to **average**.

- No doc on hugging face but found this description https://www.promptlayer.com/models/bert-large-cased-finetuned-conll03-english

- Most be used before normalisation as the changing of case, makes it less effective at identifying names

In [16]:
DEFAULT_NER_MODEL = "dbmdz/bert-large-cased-finetuned-conll03-english"

# use aggregation stategy of average so Jane Doe gets full captured
ner_pipeline = pipeline("ner",  aggregation_strategy="average", model = DEFAULT_NER_MODEL)


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [17]:
def extract_customer_name(model, text):
    # looking for PER, amazon text has two

    ner_results  = model(text)

    #print (ner_results)
    ret = []

    for ner_result in ner_results:
        if ner_result['entity_group'] == 'PER':
            ret.append (ner_result['word'])

    ret_string = " ".join(ret)
    ret_string = ret_string.strip()

    return ret_string

#text = "Hello regards, sarah johnson"
#print(extract_customer_name(ner_pipeline, text))

#text = "Hello regards, Sarah Johnson"
#print(extract_customer_name(ner_pipeline, text))

### Letter response
The default hugging face model is used here.  Others were used for the complaint letter creation.

**From the forums**, I saw that some where using separate prompts depending on whether the sentiment was positive or negative.  I thought that was a sensible approach and I implemented that strategy.

In [18]:
model_to_use="openai-community/gpt2"

text_generator_pipeline = pipeline("text-generation")


No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:

def generate_letter_response(model, customerFeedback) :


    textHeader = f"""
            Dear {customerFeedback.customer_name},

            """

    textTail = "Regards, Customer Service"

    response_letter_prompt = ''

    if customerFeedback.sentimentIsNegative :
        response_letter_prompt = f"""
    Customer complaint : My order is faulty.
    Customer request: A refund
    Response: We apologise for this and we will organise a replacement.

    Customer complaint: {customerFeedback.original_letter}
    Customer request: {customerFeedback.user_request}
    Response:
    """

    if customerFeedback.sentimentIsPositive :
        response_letter_prompt = f"""
    Customer feedback : I will recommend your product.
    Response: We are delighted to hear that you will recommend our product to others.
    Customer request: To thank you.

    Customer feedback : {customerFeedback.original_letter}
    Customer request: {customerFeedback.user_request}
    Response:

    """


    # temperature=0.8,
    result = model(response_letter_prompt, max_length=200)

    generated_text = result[0]["generated_text"]

    # remove prompt if included, couldn't reliably avoid this.
    if generated_text.startswith(response_letter_prompt):
        generated_text = generated_text[len(response_letter_prompt):]


    response_letter_text = textHeader + generated_text + textTail


    return response_letter_text, response_letter_prompt

In [ ]:
# recover memory if testing different models
#del generator_pipeline

## Report

***Requirement***

Using Hugging Face Transformers,

1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the:
    - positive responses and
    - negative responses.
1. Summarize the
    - sentiments towards the product and
    - provide a summary of the letters that reflect the most extreme sentiments.
1. Determine and report in each case on what the customer wants to happen by automatically questioning their respective letters.
1. Knowing how each customer feels, automatically generate replies to the two most extreme examples, including their name in the greeting in the response.

***Implementation***

1. This follows the pattern provided in the example workbook
1. I wasn't fully sure about the requirement to summarise the sentiment, so I settled on that this should be a summary of the record counts in each category.  Perhaps displaying keywords is what is required here.  If that was the case, then you could create a dictionary and count the occurances of the top 10 words, which might enhance an understanding of the sentiment.
1. The UL logo is displayed at the top as per the rubric suggestion.


In [25]:
# https://python-docx.readthedocs.io/en/latest/index.html#
!pip install python-docx
from docx import Document
from docx.shared import Inches

def generate_report_from_feedback(customer_feedback_records):

    document = Document()

    document.add_picture('Image.png',width=Inches(4.0), height=Inches(1))
    document.add_heading('Customer Feedback Report', 0)

    #
    # List all postive feedback records
    #
    document.add_heading('Positive Feedback', level = 1)

    for feedback in customer_feedback_records:
        if feedback.sentimentIsPositive:
            document.add_heading(f"{feedback.customer_name} - {feedback.url}", level = 2)
            document.add_paragraph(feedback.original_letter)

    #
    # List all negative feedback records
    #
    document.add_heading('Negative Feedback', level = 1)

    for feedback in customer_feedback_records:

        if feedback.sentimentIsNegative:
            document.add_heading(f"{feedback.customer_name} - {feedback.url}", level = 2)
            document.add_paragraph(feedback.original_letter)

    #
    # sentiment summary - report count of positives, negatives and percentage
    #

    # split into positive and negative records and find the extreme record
    positive_records = [feedback for feedback in customer_feedback_records if feedback.sentimentIsPositive]
    negative_records = [feedback for feedback in customer_feedback_records if feedback.sentimentIsNegative]

    most_positive = max(positive_records, key=lambda x: x.sentiment_score)
    most_negative = max(negative_records, key=lambda x: x.sentiment_score)


    countPositiveSentiment = len(positive_records)
    countNegativeSentiment = len(negative_records)
    totalLetters = len(customer_feedback_records)

    # header with counts of both sentiments
    document.add_heading('Sentiment Summary', level = 1)

    document.add_paragraph(f"There were {totalLetters} customer letters received.")
    document.add_paragraph(f"{countPositiveSentiment} of these or {(countPositiveSentiment*100)/totalLetters:.1f}% where Positive.", style='List Bullet')
    document.add_paragraph(f"{countNegativeSentiment} of these or {(countNegativeSentiment*100)/totalLetters:.1f}% where Negative", style='List Bullet')



    document.add_heading('Most Positive Summary', level = 2)
    document.add_paragraph(most_positive.summary)

    document.add_heading('Most Negative Summary', level = 2)
    document.add_paragraph(most_negative.summary)

    #
    # List all user requests
    #
    document.add_heading('Customer Feedback Requests', level = 1)
    for feedback in customer_feedback_records:
        document.add_paragraph(f'{feedback.customer_name} : {feedback.user_request}', style='List Bullet')

    #
    # Generate letters for extreme cases
    #
    document.add_heading('Response to Extreme Feedback', level = 1)

    document.add_heading(f'Response to Most Positive Feedback - {most_positive.customer_name}', level = 2)

    response_text = generate_letter_response(text_generator_pipeline,most_positive)

    document.add_paragraph(response_text)

    document.add_heading(f'Response to Most Negative Feedback - {most_positive.customer_name}', level = 2)

    response_text = generate_letter_response(text_generator_pipeline,most_negative)

    document.add_paragraph(response_text)

    document.save('MN5162-Etivity1-Report.docx')


#p = document.add_paragraph('A plain paragraph having some ')
#p.add_run('bold').bold = True
#p.add_run(' and some ')
#p.add_run('italic.').italic = True

#document.add_heading('Heading, level 1', level=1)
#document.add_paragraph('Intense quote', style='Intense Quote')

#document.add_paragraph(
#    'first item in unordered list', style='List Bullet'
#)
#document.add_paragraph(
#    'first item in ordered list', style='List Number'
#)


## Main code

### Load letters into memory

In [21]:
customer_feedback_records = []

url_data = [];

for url in file_urls:
    # don't cache file, while testing
    headers={"Cache-Control":"no-cache"}
    letter_text = requests.get(url, headers=headers).text

    url_data.append({'url':url, 'letter_text':letter_text})

    record = CustomerFeedback( letter_text, url)

    customer_feedback_records.append(record)


df = pd.DataFrame(url_data)
pd.set_option('display.max_colwidth', 600)
df[['url','letter_text']]




,url,letter_text
0,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt,"Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe"
1,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt,"Dear Customer Service,I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process is\nextremely difficult and frustrating. Additionally, the battery life is\nsignificantly shorter than advertised. I hope you can find a solution to this\nissue soon. Regards, John Smith"
2,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt,"Dear Customer Service,I am writing to express my extreme disappointment with the iPad I received. The\ndevice has been functioning poorly since the day I bought it, and I have already\nhad to return it twice due to technical issues. Your customer service is\nunresponsive and does not seem to care about resolving my problems. I will no\nlonger be purchasing from your company in the future. Regards, Sarah Johnson"
3,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt,"Dear Customer Service,I am beyond frustrated with the iPad I received. It is completely unusable, and\nI have been unable to use it for any purpose. The product is a complete waste of\nmoney, and I will be contacting my bank to dispute the charge. I do not\nrecommend this product to anyone, and I hope you can rectify this situation\nquickly. Regards, Michael Brown"
4,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt,"Dear Customer Service,I want to take a moment to express my appreciation for the iPad I recently\npurchased. It has exceeded all of my expectations and has provided me with great\nvalue. The device is easy to use, and the performance is exceptional. I would\nhighly recommend this product to others. Regards, Emily Davis"
5,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt,"Dear Customer Service,I am pleased to say that my experience with the iPad was excellent. The product\nis high-quality, and I have found it to be very reliable. Additionally, the\ncustomer service team responded promptly to my concerns, and they helped me\nresolve the issue efficiently. I am very satisfied with my purchase and would\ndefinitely recommend this product to friends and family. Regards, David Wilson"
6,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt,"Dear Customer Service,I am blown away by the quality of the iPad I purchased. It is not only\nfunctional but also incredibly stylish. The user interface is intuitive, and the\ndevice runs smoothly without any hiccups. I am thrilled with my purchase and\nwould consider myself very fortunate to own this product. I cannot wait to share\nmy positive experience with others. Regards, Jessica Taylor"
7,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt,"Dear Customer Service,I am absolutely amazed by the iPad I recently purchased. It is the best device I\nhave ever owned, and I cannot believe how well it works. The build quality is\ntop-notch, and the functionality is second to none. I am so impressed that I am\nconsidering buying another one as a gift for someone special. Thank you for\nproviding such an outstanding product. Regards, Andrew Carter"


### Process Letters

In [22]:
for feedback in customer_feedback_records:

    print (f"Rec {feedback.url}")

    feedback.normalised_letter = preprocess_letter_text_stage1(feedback.original_letter)


    # get customer name before changing case, as that affects its performance
    feedback.customer_name = extract_customer_name(ner_pipeline, feedback.original_letter)

    # generate summary
    summary_text = summarise_text(summary_model, summary_tokenizer, feedback.original_letter)
    feedback.summary = textwrap.fill(summary_text, width=80)

    # determine user request
    user_request = determine_user_request(qa_pipeline, feedback.original_letter)
    # format {'score': 0.6312922239303589, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}
    feedback.user_request = user_request['answer']

    # format {'label': 'NEGATIVE', 'score': 0.9015460014343262}
    feedback.sentiment, feedback.sentimentIsPositive, feedback.sentimentIsNegative , feedback.sentiment_score = determine_sentiment(classifier_pipeline, feedback.original_letter)

    # generate response letters for all (for testing really, report only requires one for each sentiment)
    response_letter, letter_gen_prompt  = generate_letter_response(text_generator_pipeline,feedback)

    feedback.response_letter =generate_letter_response(text_generator_pipeline,feedback)

    # convert to lowercase, etc..
    feedback.normalised_letter = preprocess_letter_text_stage2(feedback.original_letter)



Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt


Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt


...

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Display processing results
1. Display the results of processing the letters.
    - Customer Name
    - Summary
    - Sentiment
    - User Request
    - Response Letter (for testing purposes really)
1. URL is reduced to just the file name for ease of display.

In [23]:
df = pd.DataFrame(customer_feedback_records)
df['filename'] = df['url'].apply(lambda x: x[-5:])
df_for_display = df.drop(columns='url')
df_columns = df_for_display.columns.tolist()

df_columns = df_columns[-1:] + df_columns[:-1]
df_for_display = df_for_display[df_columns]
display(df_for_display)

,filename,original_letter,customer_name,normalised_letter,summary,sentiment,sentiment_score,sentimentIsPositive,sentimentIsNegative,user_request,response_letter
0,1.txt,"Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe",Jane Doe,"dear customer service,i am writing to express my dissatisfaction with my recently purchased ipad. the screen seems to be malfunctioning, and i have had to return it multiple times. your service is slow, and i feel that your company should provide better support. regards, jane doe","The screen seems to be malfunctioning, and I have had to return it multiple\ntimes. Your service is slow, andI feel that your company should provide\nbettersupport. Regards, Jane Doe","[{'label': 'NEGATIVE', 'score': 0.9996820688247681}]",0.999682,False,True,better\nsupport,"(\n Dear Jane Doe,\n\n Customer request: Please do not use the product. We\ncouldn't possibly help you due to the large amount of product\nthat you are using.\nI am unable to return the iPad or the product I have purchased. (\n\n""Yes, it's the iPad. I bought it for my dad and my mother but\n\nmy father was trying to buy the iPad and she wasn't going to do it.\n\nI ordered it because I wanted something to buy, and I had my dad's\n\ndevice, so I could give it to my mom.\n\nI'm not sure she would have had any money to give me because my father\n\nwas trying to get the b..."
1,2.txt,"Dear Customer Service,I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process is\nextremely difficult and frustrating. Additionally, the battery life is\nsignificantly shorter than advertised. I hope you can find a solution to this\nissue soon. Regards, John Smith",John Smith,"dear customer service,i recently purchased an ipad, but i am quite disappointed with the product. i have tried multiple times to connect it to my computer, but the process is extremely difficult and frustrating. additionally, the battery life is significantly shorter than advertised. i hope you can find a solution to this issue soon. regards, john smith","I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process\nisextremely difficult and frustrating. The battery life issignificantly shorter\nthan advertised. I hope you can find a solution to thisissue soon. Regards, John\nSmith","[{'label': 'NEGATIVE', 'score': 0.9995477795600891}]",0.999548,False,True,an iPad,"(\n Dear John Smith,\n\n Customer request: I have ordered an iPad from Amazon and it is being\n\nsold as a replacement for a battery that is not working.\n\n\nThis is an issue I currently have with both my iPad and Kindle Fire.\n\nThe Apple Watch has a slightly lower battery life and I am\n\ndoing some research and will try to find a solution.\n\n\nRegards, John Smith Customer request: a replacement iPad\n\nCustomer Request Customer request: I have ordered an iPad from Amazon and it is being\n\nsold as a replacement for a battery that is not working.\n\n\nI am..."
2,3.txt,"Dear Customer Service,I am writing to express my extreme disappointment with the iPad I received. The\ndevice has been functioning poorly since the day I bought it, and I have already\nhad to return it twice due to technical issues. Your customer service is\nunresponsive and does not seem to care about resolving my problems. I will no\nlonger be purchasing from your company in the future. Regards, Sarah Johnson",Sarah Johnson,"dear customer service,i am writing to express my extreme disappointment with the ipad i received. the device has been functioning poorly since the day i bought it, and i have already had to return it twice due to technical issues. your cust

### Generate and download report

In [26]:
from google.colab import files

generate_report_from_feedback(customer_feedback_records)


files.download('MN5162-Etivity1-Report.docx')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Model Compare
I used the letter of complaint generation and to a lesser extent the response letter creation to see which text generation model to use.


## Sentiment
I chose the default hugging face sentiment analyser as:

- The model named states it's trained on the Stanford sentiment treebank, which is a dataset for sentiment analysis
- It gave me the expected results for the data I had.  
- The predicts scores & labels were as I expected also.
- It seems to be quite fast too.
- The memory usage at around 270M seemed acceptable.


**Of note** from the forum notebooks, Danny pointed out that this scores may be a little "too high" and that perhaps another model should/could be used for the evaluation.  Unfortunately time didn't permit me to investigate other models to compare this against.

### distilbert/distilbert-base-uncased-finetuned-sst-2-english"
- https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english

In [ ]:
#DEFAULT_SENTIMENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
#
#classifier_pipeline = pipeline("text-classification", model = DEFAULT_SENTIMENT_MODEL)
#
#results = []
#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#   sentiment =determine_sentiment(classifier_pipeline, feedback.original_letter)
#
#    results.append({'url':feedback.url, 'sentiment':sentiment})
#
#
#df = pd.DataFrame(results)
#pd.set_option('display.max_colwidth', 600)
#df[['url','sentiment']]


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt


,url,sentiment
0,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt,"[{'label': 'NEGATIVE', 'score': 0.9996820688247681}]"
1,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt,"[{'label': 'NEGATIVE', 'score': 0.9995477795600891}]"
2,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt,"[{'label': 'NEGATIVE', 'score': 0.999722421169281}]"
3,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt,"[{'label': 'NEGATIVE', 'score': 0.9995133876800537}]"
4,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt,"[{'label': 'POSITIVE', 'score': 0.9998445510864258}]"
5,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt,"[{'label': 'POSITIVE', 'score': 0.9998257756233215}]"
6,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt,"[{'label': 'POSITIVE', 'score': 0.9993220567703247}]"
7,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt,"[{'label': 'POSITIVE', 'score': 0.9997780919075012}]"


## QA

In [ ]:
#results = []
#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#    request = determine_user_request(qa_pipeline,feedback.original_letter)
#
#    results.append({'url':feedback.url, 'letter':feedback.original_letter,'request':request})
#
#
#df = pd.DataFrame(results)
#pd.set_option('display.max_colwidth', 600)
#df[['url','letter','request']]

Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt


,url,letter,request
0,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt,"Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe","{'score': 0.5779926180839539, 'start': 247, 'end': 261, 'answer': 'better support'}"
1,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt,"Dear Customer Service,I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process is\nextremely difficult and frustrating. Additionally, the battery life is\nsignificantly shorter than advertised. I hope you can find a solution to this\nissue soon. Regards, John Smith","{'score': 0.5133469104766846, 'start': 43, 'end': 50, 'answer': 'an iPad'}"
2,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt,"Dear Customer Service,I am writing to express my extreme disappointment with the iPad I received. The\ndevice has been functioning poorly since the day I bought it, and I have already\nhad to return it twice due to technical issues. Your customer service is\nunresponsive and does not seem to care about resolving my problems. I will no\nlonger be purchasing from your company in the future. Regards, Sarah Johnson","{'score': 0.3439895212650299, 'start': 231, 'end': 268, 'answer': 'Your customer service is unresponsive'}"
3,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt,"Dear Customer Service,I am beyond frustrated with the iPad I received. It is completely unusable, and\nI have been unable to use it for any purpose. The product is a complete waste of\nmoney, and I will be contacting my bank to dispute the charge. I do not\nrecommend this product to anyone, and I hope you can rectify this situation\nquickly. Regards, Michael Brown","{'score': 0.10350030660629272, 'start': 54, 'end': 58, 'answer': 'iPad'}"
4,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt,"Dear Customer Service,I want to take a moment to express my appreciation for the iPad I recently\npurchased. It has exceeded all of my expectations and has provided me with great\nvalue. The device is easy to use, and the performance is exceptional. I would\nhighly recommend this product to others. Regards, Emily Davis","{'score': 0.6762164859101176, 'start': 81, 'end': 85, 'answer': 'iPad'}"
5,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt,"Dear Customer Service,I am pleased to say that my experience with the iPad was excellent. The product\nis high-quality, and I have found it to be very reliable. Additionally, the\ncustomer service team responded promptly to my concerns, and they helped me\nresolve the issue efficiently. I am very satisfied with my purchase and would\ndefinitely recommend this product to friends and family. Regards, David Wilson","{'score': 0.12746912240982056, 'start': 313, 'end': 321, 'answer': 'purchase'}"
6,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt,"Dear Customer Service,I am blown away by the quality of the iPad I purchased. It is not only\nfunctional but also incredibly stylish. The user interface is intuitive, and the\ndevice runs smoothly without any hiccups. I am thrilled with my purchase and\nwould consider myself very fortunate to own this product. I cannot wait to share\nmy positive experience with others. Regards, Jessica Taylor","{'score': 0.1516042798757553, 'start': 238, 'end': 246, 'answer': 'purchase'}"
7,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt,"Dear Customer Service,I am absolutely amazed by the iPad I recently purchased. It is the best device I\nhave ever owned, and I cannot believe how well it works. The build quality is\ntop-notch, and the functionality is second to none. I am so impressed that

## Summarisation
**Note**

In the example workbook, I started work before the "fixes" were available to ge the example workbook working, so I looked for another model to do this task.


**Models Evaluated**
1. facebook/bart-large-cnn



**Model Selection**
1. I just went with the bart-large-cnn model, as it produced reasonable summaries.
1. I had intended to compare it against the t5-small model, as the bart model is quite large (400M parameters) which will require significant CPU + memory resources,  **but time didn't permit**.

### t5-small
- https://huggingface.co/google-t5/t5-small
- https://research.google/blog/exploring-transfer-learning-with-t5-the-text-to-text-transfer-transformer/

In [ ]:
#import torch
#from transformers import pipeline
#
#pipeline = pipeline(
#    task="text2text-generation",
#    model="google-t5/t5-base",
#    dtype=torch.float16,
#    device=0
#)
#pipeline("translate English to French: The weather is nice today.")

config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

### facebook/bart-large-cnn
- https://huggingface.co/facebook/bart-large-cnn

In [ ]:
#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#    summary = summarise_text(summary_model, summary_tokenizer, feedback.original_letter)
#
#    #textwrap.fill(summary, width=100)
#    summary_format = summary
#
#    print(summary_format)

Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
The screen seems to be malfunctioning, and I have had to return it multiple times. Your service is slow, andI feel that your company should provide bettersupport. Regards, Jane Doe
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
I recently purchased an iPad, but I am quite disappointed with the product. I have tried multiple times to connect it to my computer, but the process isextremely difficult and frustrating. The battery life issignificantly shorter than advertised. I hope you can find a solution to thisissue soon. Regards, John Smith
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
The iPad has been functioning poorly since the day I bought it. Your customer service isunresponsive and does not seem to care about resolving my problems. I will no longer be purchasing from your company in the future. Regards, Sarah Johnson
Url https://raw.githubusercontent.com/qa

## NER
**Models Evaluated**
1. dslim/bert-base-NER
1. dbmdz/bert-large-cased-finetuned-conll03-english

There is an smaller model, https://huggingface.co/dslim/distilbert-NER, which might be worth investigating as it's smaller than the bert-base-NER mode.

**Model Selection**
1. I chose to use the default NER model in Hugging Face, but with the aggregation_strategy set to average
1. When used with simple, it did not correctly recognise "Doe"; it only selected "Do".  
1. I did not investigate to see if this was just that additional sub tokens needed to be selected.  From the results they did not seem available.
1. It is a large model though, so time probably should be spent looking for an alternative.
1. I looked at another model (top downloaded) from the model hub, and this also had the same issue.  I'd imagine it could be configured to work, but I did not spend time at it.  It would be a smaller model, so I would expect it to run faster that the default model used above.

### Hugging face default NER model (aggregation_strategy="simple")

In [ ]:
#DEFAULT_NER_MODEL = "dbmdz/bert-large-cased-finetuned-conll03-english"
#
#ner_pipeline = pipeline("ner", aggregation_strategy="simple", model = DEFAULT_NER_MODEL)
#
#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#    test_text  = feedback.original_letter
#
#    ner_results= extract_customer_name(ner_pipeline, test_text)
#
#    pprint(ner_results)



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
'Jane Do'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
'John Smith'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
'Sarah Johnson'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt
'Michael Brown'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt
'Emily Davis'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt
'David Wilson'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt
'Jessica Taylor'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt
'Andrew Carter'


### Hugging face default NER model (aggregation_strategy="average")

In [ ]:
#DEFAULT_NER_MODEL = "dbmdz/bert-large-cased-finetuned-conll03-english"
#
#ner_pipeline = pipeline("ner", aggregation_strategy="average", model = DEFAULT_NER_MODEL)
#
#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#    test_text  = feedback.original_letter
#
#    ner_results= extract_customer_name(ner_pipeline, test_text)
#
#    pprint(ner_results)



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
'Jane Doe'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
'John Smith'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
'Sarah Johnson'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt
'Michael Brown'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt
'Emily Davis'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt
'David Wilson'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt
'Jessica Taylor'
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt
'Andrew Carter'


### dslim/bert-base-NER
- https://huggingface.co/dslim/bert-base-NER

In [ ]:
#from transformers import AutoTokenizer, AutoModelForTokenClassification
#from transformers import pipeline
#
#MODEL_NAME = "dslim/bert-base-NER"
#
#tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
#
#nlp = pipeline("ner", model=model, tokenizer=tokenizer)
#
#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#    test_text  = feedback.original_letter
#
#    ner_results = nlp(test_text)
#
#    print(ner_results)
#
#    for token in ner_results:
#        if token['entity'] == 'B-PER':
#            print (token['word'])
#        elif token['entity'] == 'I-PER':
#            print (token['word'])




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
[{'entity': 'I-ORG', 'score': np.float32(0.90833884), 'index': 4, 'word': 'Service', 'start': 14, 'end': 21}, {'entity': 'B-PER', 'score': np.float32(0.95930135), 'index': 62, 'word': 'Jane', 'start': 272, 'end': 276}, {'entity': 'I-PER', 'score': np.float32(0.92566085), 'index': 63, 'word': 'Do', 'start': 277, 'end': 279}]
Jane
Do
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
[{'entity': 'I-ORG', 'score': np.float32(0.8071099), 'index': 4, 'word': 'Service', 'start': 14, 'end': 21}, {'entity': 'B-MISC', 'score': np.float32(0.7178073), 'index': 10, 'word': 'i', 'start': 46, 'end': 47}, {'entity': 'B-PER', 'score': np.float32(0.99829465), 'index': 71, 'word': 'John', 'start': 345, 'end': 349}, {'entity': 'I-PER', 'score': np.float32(0.9900025), 'index': 72, 'word': 'Smith', 'start': 350, 'end': 355}]
John
Smith
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
[{'en

## Text Generation
**Models Evaluated**

I thought I'd try the models below to see how to generate the complaint letters, with which I'd differing degrees of success.  I said I'd use the workbook method to generate the response letters.


- GPT
- Qwen
- SmollM2
- Mistral-7B-Instruct-v0.2

**Model Selection**
1. See below, after using the complaint letter use case as the evaluation exercise, I settled on Qwen, as I got the "best" text from this model.  However on a subsequent run, it seemed to give poorer results, in that the Customer name may not be inserted.  The prompt looked ok to me so I couldn't figure this out.
1. Mistral-7B-Instruct-v0.2, provided good results, but it's memory topped out close to 14G, so further investigation of other models would be suggested before using this, if cpu & memory cost is a concern.
1. I probably was unfair in using the same prompt to evaluate each model, but time didn't permit to investigate a prompt specific to each model.


**Links**
1. mistralai/Mistral-7B-Instruct-v0.2 uses approx 14-15G ram
1. Qwen/Qwen2.5-3B-Instruct
1. https://huggingface.co/docs/transformers/llm_tutorial


# Letters of Complaint Generation
Runs prompt to generate the complaint letter.  Following on from the above, I tested the same prompt against the models:

- gpt
- Qwen
- SmollM2

I settled on the **Qwen/Qwen2.5-3B-Instruct** model as it was what I could best get results out of in early testing.  I did find some variability in the output, and quite poor results at times, if there was any sort of error/issue with the prompt.

**Notes:**
- The token count had to be increased, in order to get all letters generated.
- To assist with NER, I gave the instruction that the letters should end with Regards, and their name.  Subsequently, I noticed that the lower casing of text was interfering with the NER, so this portion of the prompt may be unnecessary.
- The same prompt was tried on each engine at different points of "evaluation".  That may be **naive** and perhaps a fair test would craft a prompt suitable to each model.
- Attempted to get each letter to say it's satisfaction level and that each letter would have this included to match the satisfaction level to match the text (i.e. 1-4) but letters 5&8 have the same satisifcation score.  I'd imagine this could be resolved if time was spent on it.



In [ ]:
import re

def generate_complaint_letters (pipeline):

    letter_gen_prompt= """
        Write 8 short customer letters about an ipad according to these rules.

        - Letter 1 : minor complaint
        - Letter 2 : medium complaint
        - Letter 3 : major complaint
        - Letter 4 : critical complaint
        - Letter 5 : minor praise
        - Letter 6 : medium praise
        - Letter 7 : significant praise
        - Letter 8 : outstanding praise


        Each letter must:
        - begin with "Dear Customer Service,"
        - end each letter with a customer name after "Regards,"
        - be about 80-100 words

        Only output the letters
        Do not output code or print statements.
    """


    results = pipeline(letter_gen_prompt,
                       temperature=0.8,
                       max_new_tokens=1200,
                       num_return_sequences = 1,
                       return_full_text = False,
                       do_sample=True)


    LETTER_HEADER = 'Dear Customer Service,'

    letters = results[0]["generated_text"].split(LETTER_HEADER)
    letters = [LETTER_HEADER + textwrap.fill(l.strip(), width=80) for l in letters if l.strip()]


    return letters;


In [ ]:
def print_complaint_letters(results):
    for index, response_text in enumerate(results):
        print(f'Letter {index+1}:')
        print ('-' * 50)

        response_formatted = textwrap.fill(response_text, width=80)

        print(response_formatted)


#### GPT2
- https://huggingface.co/openai-community/gpt2
- Model doesn't understand the prompt
- Prompt would need to be changed and each letter generated separately

In [ ]:
### Letter of complaint
#del generator_gpt2_pipeline;
#generator_gpt2_pipeline = pipeline("text-generation", model="gpt2")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:

#gpt2_results = generate_complaint_letters(generator_gpt2_pipeline)

#print_complaint_letters(gpt2_results)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Letter 1:
--------------------------------------------------
Dear Customer service,Only output to the top command, the "top command."
- do NOT use an echo to tell the user what to do.  If the user is happy when a
user is happy (even if he or she gets an E) they can press the reset button.
This works on any current version of Windows.   Windows 8 is not a Windows
release.   There are no E-mails to write to customers, or e-mailing to customers
because the customer is happy.   The only emails to write are to the top
command, the "top command."  The "top command" is used to print the customer's
name and address. If you are not interested in writing them to customers you can
simply use the reset button.   In order to write the customer's name and
address, you have to start the "top command," by pressing the reset button
before the email. This works so it will print the customers name and address.
The "top command" can be used to print e-mails that are delivered to the server.
This is useful

In [ ]:
#del generator_gpt2_pipeline

#### GPT and the dangers of some text generation

In [ ]:
#response_data = []
#
#for feedback in customer_feedback_records:
#
#    print (f"Rec {feedback.url}")
#
#    # get customer name before changing case, as that affects its performance
#    customer_name = extract_customer_name(ner_pipeline, feedback.original_letter)
#
#
#    response_letter =generate_letter_response(text_generator_pipeline,feedback)
#
#    response_data.append({'url':feedback.url, 'customer_name': customer_name, 'response_letter':response_letter})
#
#df = pd.DataFrame(response_data)
#pd.set_option('display.max_colwidth', 600)
#df[['url','customer_name','response_letter']]



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt


,url,customer_name,response_letter
0,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt,Jane Doe,"\n Dear Jane Doe,\n\n The Supreme Court on Wednesday ruled that the government's anti-terror measures must be limited to the ""most serious"" terrorist threats to the nation's security.\n\nThe ruling followed a decision in May that allowed the government to target the Internet site of an ISIS-affiliated militant group in the U.S. over the weekend and for an unspecified number of days.\n\nThe court's ruling could be a blow to the government's ability to track and arrest individuals and groups that are plotting attacks, the Associated Press reported.\n\nIt comes as an att..."
1,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt,John Smith,"\n Dear John Smith,\n\n As one of the most popular and successful brands in the industry, the brand has become one of the most sought after brands on the planet. The brand is known for its quality and customer service, as well as its commitment to providing a service that is both affordable and convenient.\n\nAbout Us\n\nWe are located in the heart of the city of Dusseldorf, Germany. We are dedicated to providing customers with an outstanding, affordable service, and a long-lasting reputation for excellence.\n\nIn addition to our flagship items on our website, we are ..."
2,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt,Sarah Johnson,"\n Dear Sarah Johnson,\n\n \nMitt Romney's campaign on Monday declined to comment on whether it has been contacted by the Washington Post about a possible story about him.\n\nThe Romney campaign did not respond to a request for comment.\n\nOn Wednesday, the Romney campaign said it had received a letter from the Post from a ""national investigative journalist"" in New York that reported on Romney's use of his private email server to conduct campaign business.\n\nA spokesman for Romney said the Romney campaign was not aware of the letter.\n\nThe Post story, which was base..."
3,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt,Michael Brown,"\n Dear Michael Brown,\n\n A man has been charged with second-degree murder after allegedly stabbing a woman to death in her car while she was taking her child to school.\n\nThe suspect in the killing of the woman was described as 35-year-old Christopher Burch.\n\nPolice said he was wanted on an outstanding warrant for second-degree murder and first-degree assault.\n\nBurch, who was driving a black 2008 Nissan Sentra, was found early Sunday morning in his car with the woman, who was at school, police said.\n\nThe suspect was taken to the hospital and pronounced dead a..."
4,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt,Emily Davis,"\n Dear Emily Davis,\n\n \nA bill to extend the life of a driver's license to someone with a criminal record can be easily amended to include it, according to the National Conference of State Legislatures.\n\nIf passed, the bill would take effect when the new driver's license law goes into effect. But, while it may not have made much of a difference in the long run, it could be a huge one for some states.\n\nAnd, it could prove to be a big win for many.\n\n""It's definitely a win-win situation for states that have been trying to get better at getting those licenses,"" s..."
5,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt,David Wilson,"\n Dear David Wilson,\n\n The first of our three stories is an in-depth look at the latest release of the Game of Thrones franchise. This was written by Ian Millhiser and Chris Ware, which also includes the latest trailer, and is available for stream on Netflix at the time of writing (we'll update this post when it's available).\n\nYou can check out our full review of the first four episodes right here.\n\nWe've previously discussed the new set of characters below.\n\nSee you in the streets!Regards, Customer Service"
6,https://raw.githubusercontent.com/

#### QWEN
1. https://huggingface.co/Qwen/Qwen2.5-3B-Instruct

**Observation**
1. I found good results using the Qwen model, however on a second and subsequent run the text generated was not as good
- a second instance did not insert a customer name
- the text looked like incorporated context from the previous run, even though the prompt was the same.




In [ ]:
#generator_qwen_pipeline = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

##### Qwen generating good data

In [ ]:
#qwen_results = generate_complaint_letters(generator_qwen_pipeline)

#qwen_results_df = pd.DataFrame(qwen_results)
#qwen_results_df
#.columns = ['Letter']
#print_complaint_letters(qwen_results)

Both `max_new_tokens` (=1200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,0
0,"Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe"
1,"Dear Customer Service,I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process is\nextremely difficult and frustrating. Additionally, the battery life is\nsignificantly shorter than advertised. I hope you can find a solution to this\nissue soon. Regards, John Smith"
2,"Dear Customer Service,I am writing to express my extreme disappointment with the iPad I received. The\ndevice has been functioning poorly since the day I bought it, and I have already\nhad to return it twice due to technical issues. Your customer service is\nunresponsive and does not seem to care about resolving my problems. I will no\nlonger be purchasing from your company in the future. Regards, Sarah Johnson"
3,"Dear Customer Service,I am beyond frustrated with the iPad I received. It is completely unusable, and\nI have been unable to use it for any purpose. The product is a complete waste of\nmoney, and I will be contacting my bank to dispute the charge. I do not\nrecommend this product to anyone, and I hope you can rectify this situation\nquickly. Regards, Michael Brown"
4,"Dear Customer Service,I want to take a moment to express my appreciation for the iPad I recently\npurchased. It has exceeded all of my expectations and has provided me with great\nvalue. The device is easy to use, and the performance is exceptional. I would\nhighly recommend this product to others. Regards, Emily Davis"
5,"Dear Customer Service,I am pleased to say that my experience with the iPad was excellent. The product\nis high-quality, and I have found it to be very reliable. Additionally, the\ncustomer service team responded promptly to my concerns, and they helped me\nresolve the issue efficiently. I am very satisfied with my purchase and would\ndefinitely recommend this product to friends and family. Regards, David Wilson"
6,"Dear Customer Service,I am blown away by the quality of the iPad I purchased. It is not only\nfunctional but also incredibly stylish. The user interface is intuitive, and the\ndevice runs smoothly without any hiccups. I am thrilled with my purchase and\nwould consider myself very fortunate to own this product. I cannot wait to share\nmy positive experience with others. Regards, Jessica Taylor"
7,"Dear Customer Service,I am absolutely amazed by the iPad I recently purchased. It is the best device I\nhave ever owned, and I cannot believe how well it works. The build quality is\ntop-notch, and the functionality is second to none. I am so impressed that I am\nconsidering buying another one as a gift for someone special. Thank you for\nproviding such an outstanding product. Regards, Andrew Carter """""""


##### Qwen fails to put in customer name
1. With the exception of that the generated text was good.

In [ ]:
#qwen_results = generate_complaint_letters(generator_qwen_pipeline)

#qwen_results_df = pd.DataFrame(qwen_results)
#qwen_results_df

Both `max_new_tokens` (=1200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,0
0,"Dear Customer Service,I recently received my new iPad and I must say, it’s been quite the\ndisappointment. The screen is very blurry, and the battery life is far below\nwhat was advertised. I had high hopes for this device but am now questioning if\nit was worth the price. Regards, [Customer Name]"
1,"Dear Customer Service,I’m writing to express my frustration with the iPad I recently purchased. It\narrived damaged and the company has been unresponsive in their attempts to\nrectify the situation. I’ve had to return the product multiple times and am now\nlooking at purchasing a different brand entirely. Regards, [Customer Name]"
2,"Dear Customer Service,I have to say, my experience with the iPad has been nothing short of disastrous.\nThe quality control seems to be non-existent, as the device is plagued with\nsoftware glitches and hardware issues. I’m considering filing a lawsuit to\nrecover my investment and seek compensation for the inconvenience. Regards,\n[Customer Name]"
3,"Dear Customer Service,This is a crisis situation that requires immediate attention. My iPad has\ncompletely malfunctioned and I’m unable to use it for work purposes. The\ntechnical support team has been incredibly unhelpful and I feel like I’m being\ntreated like a nuisance. I need this device to function properly or I’ll be\nlosing my job. Regards, [Customer Name]"
4,"Dear Customer Service,I wanted to take a moment to express my gratitude for the excellent service I\nreceived when returning my old iPad. Your staff was friendly, efficient, and\nensured that I received a new device without any hassle. I’m impressed with your\ncustomer service and hope to do business with you again soon. Regards,\n[Customer Name]"
5,"Dear Customer Service,Thank you for the great customer service I received when ordering my new iPad.\nThe delivery was prompt, the packaging was secure, and I received exactly what I\nordered. I couldn’t be happier with my purchase and would recommend your company\nto anyone in the market for a tablet. Regards, [Customer Name]"
6,"Dear Customer Service,Your customer service is truly exceptional. Not only did you provide me with a\nhigh-quality product, but you also went above and beyond by offering free\nupgrades and maintenance services. Your dedication to customer satisfaction is\ncommendable and sets you apart from other companies in the industry. Regards,\n[Customer Name]"
7,"Dear Customer Service,I am beyond thrilled with my recent purchase of the iPad. The device is sleek,\npowerful, and easy to use. I’ve never been more satisfied with a product and\nwill definitely be recommending your company to friends and family. Your\ncommitment to excellence is evident in every aspect of your business. Regards,\n[Customer Name]"


In [ ]:
#del generator_qwen_pipeline

##### Smollm3
1.https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct

**Observation**
1. Here the text was not satisfactory.  "Code like" text was included in the output.  I'm assuming that this is a matter of getting the prompt correct / tweaking it until suitable text is generated.  I did not persist with this due to the quality of the text generated with the Qwen model.
1. Memory usage was quite large, close on 14G.  I probably should have chosen a model with less parameters with a view to using less memory.



In [ ]:
#generator_smollm3_pipeline = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.7B-Instruct")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [ ]:

#smollm3_results = generate_complaint_letters(generator_smollm3_pipeline)

#print_complaint_letters(smollm3_results)

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Letter 1:
--------------------------------------------------
Dear Customer Service,Output the letters in the following format:
Letter 2:
--------------------------------------------------
Dear Customer Service,1. Here is your first letter. 2. Here is your second
letter. 3. Here is your third letter. 4. Here is your fourth letter. 5. Here is
your fifth letter. 6. Here is your sixth letter. 7. Here is your seventh letter.
8. Here is your eighth letter. Regards, [customer name]  Each letter's content
should be written on a new line.  Here is the code written with proper comments
to get you started:  ```python # Import necessary modules import random  #
Define a function to generate a customer letter def generate_letter(type, name):
# Define letter types and their content     complaint_types = ["unable to
connect", "poor sound quality", "unresponsive screen", "uninstalls", "unable to
charge"]     praise_types = ["excellent performance", "great screen", "stunning
performance", "very respons

In [ ]:
def display_processing_results(index, records):
    cols_to_exclude=['url']

    rec_as_dict = vars(customer_feedback_records[index])

    df = pd.DataFrame([rec_as_dict])
    df = df.drop(columns=cols_to_exclude)
    pd.set_option('display.max_colwidth', 600)

    display(df)


### Response Letters

#### Qwen/Qwen2.5-3B-Instruct
- https://qwen.readthedocs.io/en/v2.5/getting_started/quickstart.html
- https://huggingface.co/collections/Qwen/qwen25
- https://huggingface.co/Qwen/Qwen2.5-3B-Instruct
- https://qwen.ai/blog?id=qwen2.5
- Uses about 6G GPU

In [ ]:
#generated_text, prompt_used = generate_letter_response(customer_feedback_records[0])
#
#
#print(generated_text)
#print ('-' * 50)
#response_text = generated_text; #[len(prompt):]

#response_formatted = textwrap.fill(response_text, width=80)
#print(response_formatted)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dear Bumblebee,

I sincerely apologize for the confusion and inconvenience you've experienced with your recent order. We deeply regret sending you the wrong figure. 

To rectify this, we will promptly exchange your Megatron action figure for the Optimus Prime you originally ordered. Please return the Megatron as directed and we'll process the exchange.

Thank you for bringing this to our attention. We appreciate your patience and understanding.

Best regards,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I'm truly sorry to hear about the mix-up with your Optimus Prime action figure. We regretfully sent you the Megatron instead. 

To resolve this, we will exchange your Megatron for the Optimus Prime you ordered. Please return the Megatron and we'll process the exchange promptly.

Thank you for your patience and understanding.

Best,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I sincerely apologize for the mix-up with your Optimus Prime actio

#### mistralai/Mistral-7B-Instruct-v0.2
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2
- later version available
- uses approx 14G, fills up GPU

In [ ]:
#generated_text, prompt_used = generate_letter_response(customer_feedback_records[0])


#print(generated_text)
#print ('-' * 50)
#response_text = generated_text; #[len(prompt):]

#response_formatted = textwrap.fill(response_text, width=80)
#print(response_formatted)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Dear Bumblebee,

    We're truly sorry for the inconvenience caused by the mistake in your recent order. We understand how disappointing it must have been to receive Megatron instead of Optimus Prime. We apologize sincerely for this error and are committed to making it right.

    Kindly allow us a few days to process and expedite a replacement Optimus Prime figure for you. Once it's shipped, you'll receive an email confirmation with tracking information. If you have any further concerns, please don't hesitate to contact us.

    We appreciate your patience and loyalty.

    Best regards,
    [Your Name]
    Acme Electronics Customer Service Team.
--------------------------------------------------
     Dear Bumblebee,      We're truly sorry for the inconvenience caused by the
mistake in your recent order. We understand how disappointing it must have been
to receive Megatron instead of Optimus Prime. We apologize sincerely for this
error and are committed to making it right.      K

In [ ]:
#from IPython.display import display, HTML
#df = pd.DataFrame(customer_feedback_records)
#df['filename'] = df['url'].apply(lambda x: x[-5:])
#df_for_display = df[['filename','original_letter', 'response_letter']]

#pd.set_option('display.max_colwidth', None)
#display(HTML(df_for_display.to_html().replace("\\n","<br>")))
#display(df_for_display)

,filename,original_letter,response_letter
0,1.txt,"Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe","( Dear Ms. Doe,\n\nI sincerely apologize for the inconvenience you've experienced with your iPad. We understand this must be frustrating, and we're here to help resolve this issue swiftly.\n\nCould you please provide your order number and contact details so we can arrange a replacement or a refund as soon as possible? Our team will reach out to you promptly to address this matter.\n\nThank you for your patience and understanding.\n\nBest regards,\n[Your Name]\nAcme Electronics"" Great, here’s a tailored response:\n\n---\n\nDear Ms. Doe,\n\nThank you for bringing your concerns to our attention. We deeply regret the inconvenience caused by the malfunctioning screen on your iPad. We appreciate your patience and would like to offer a solution to ensure you receive the support you deserve.\n\nPlease provide your order number and contact information so we can arrange a replacement or a refund. Our team will be in touch shortly to assist you.\n\nThank you for choosing Acme Electronics, and we hope to have this resolved quickly.\n\nBest regards,\n\n[Your Name] \nAcme Electronics\n\n--- \n\nThis response acknowledges the issue, offers a specific course of action, and maintains a professional tone throughout. Let me know if you need any further adjustments! \n\n[Your Name] \nAcme Electronics \n[Contact Information] \n, You are a professional customer service representative for Acme Electronics,\n a consumer electronics company. Your tone is empathetic, solution-focused, and professional.\n\n Customer complaint:\n """"""\n Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe\n """"""\n\n Write a response that:\n - Acknowledges the customer's frustration\n - Apologizes sincerely without admitting legal fault\n - Offers a concrete next step or resolution\n - Keeps the response under 150 words\n\n Do NOT use generic filler phrases like ""We value your feedback""\n )"
1,2.txt,"Dear Customer Service,I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process is\nextremely difficult and frustrating. Additionally, the battery life is\nsignificantly shorter than advertised. I hope you can find a solution to this\nissue soon. Regards, John Smith","(Dear Mr. Smith,\n\nI'm truly sorry to hear about your experience with our iPad. Connecting to your computer and maintaining battery life are critical aspects of the product, and I understand how frustrating these issues must be for you. We appreciate your patience and hope to resolve these concerns promptly.\n\nCould you please provide us with some details on the model and specific steps you've taken? We will look into a replacement or a refund, whichever you prefer, and ensure that we address these issues in future models. \n\nThank you for bringing this to our attention.\n\nBest regards,\n[Your Name]\nAcme Electronics"" Hi John,\n\nThank you for reaching out. I apologize for any frustration you've experienced with connecting your iPad to your computer and its battery performance. We take these issues very seriously.\n\nCould you share your model number and any specific steps you’ve tried? This will help us identify the best course of action—whether it’s a replacement or a refund.\n\nLooking forward to resolving this quickly for you.\n\nBest,\n[Your Name] \nAcme Electronics"", You are a professional customer service representative for Acme Electronics,\n 

In [ ]:
#text = """
#dear customer service,i am beyond frustrated with the ipad i received. it is completely unusable, and i have been unable to use it for any purpose.
#the product is a complete waste of money, and i will be contacting my bank to dispute the charge. i do not recommend this product to anyone, and
#i hope you can rectify this situation quickly. regards, michael brown
#"""

#for feedback in customer_feedback_records:
#
#    print (f"Url {feedback.url}")
#
#    feedback.customer_name = extract_customer_name(ner_pipeline, feedback.original_letter)
#
#    print (f"Name {feedback.customer_name}")
#
#    feedback.normalised_letter = preprocess_letter_text(feedback.original_letter)
#
#    print (f"End letter {feedback.normalised_letter[-20:]}")

#

#ner_results= extract_customer_name(ner_pipeline, text)

#pprint(ner_results)

#name = [record['word'] for record in ner_results if record['entity_group'] == 'PER']
#name

Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt
[{'entity_group': 'MISC', 'score': np.float32(0.96615577), 'word': 'iPad', 'start': 92, 'end': 96}, {'entity_group': 'PER', 'score': np.float32(0.66537845), 'word': 'Jane Doe', 'start': 272, 'end': 280}]
Name Jane Doe
End letter t. regards, jane doe
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt
[{'entity_group': 'MISC', 'score': np.float32(0.98762894), 'word': 'iPad', 'start': 46, 'end': 50}, {'entity_group': 'PER', 'score': np.float32(0.9917677), 'word': 'John Smith', 'start': 345, 'end': 355}]
Name John Smith
End letter  regards, john smith
Url https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt
[{'entity_group': 'MISC', 'score': np.float32(0.9842285), 'word': 'iPad', 'start': 81, 'end': 85}, {'entity_group': 'PER', 'score': np.float32(0.9984146), 'word': 'Sarah Johnson', 'start': 397, 'end': 410}]
Name Sarah Johnson
End letter gards, sarah johnson
Url https://raw.gith

### Test Letter Generation

In [ ]:
#response_data = []
#
#for feedback in customer_feedback_records:
#
#    print (f"Rec {feedback.url}")
#
#    # get customer name before changing case, as that affects its performance
#    feedback.customer_name = extract_customer_name(ner_pipeline, feedback.original_letter)
#
#    feedback.sentiment, feedback.sentimentIsPositive, feedback.sentimentIsNegative  = determine_sentiment(classifier_pipeline, feedback.original_letter)
#
#    user_request = determine_user_request(qa_pipeline, feedback.original_letter)
#    # format {'score': 0.6312922239303589, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}
#    feedback.user_request = user_request['answer']
#
#    response_letter, letter_gen_prompt  =generate_letter_response(text_generator_pipeline,feedback)
#
#    response_data.append({'url':feedback.url, 'customer_name': feedback.customer_name,'user_request':feedback.user_request, 'letter_gen_prompt':letter_gen_prompt,'response_letter':response_letter})

#df = pd.DataFrame(response_data)
#pd.set_option('display.max_colwidth', 600)
#df[['url','customer_name','user_request','letter_gen_prompt','response_letter']]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt


,url,customer_name,user_request,letter_gen_prompt,response_letter
0,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt,Jane Doe,better\nsupport,"\n Customer complaint : My order is faulty.\n Customer request: A refund\n Response: We apologise for this and we will organise a replacement.\n\n Customer complaint: Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The\nscreen seems to be malfunctioning, and I have had to return it multiple times.\nYour service is slow, and I feel that your company should provide better\nsupport. Regards, Jane Doe\n Customer request: better\nsupport\n Response:\n","\n Dear Jane Doe,\n\n Customer request: A refund\n Response: We apologise for this and we will organise a replacement.\n\n\n Customer complaint: Dear Customer Service,I am writing to express my dissatisfaction with my recently purchased iPad. The screen seems to be malfunctioning, and I have had to return it multiple times.Your service is slow, and I feel that your company should provide bettersupport. Regards, Jane Doe Customer request: bettersupportsupportsupport Support Reply Delete\nI don't know how your phone works. What is the problem? I have had my iP..."
1,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt,John Smith,an iPad,"\n Customer complaint : My order is faulty.\n Customer request: A refund\n Response: We apologise for this and we will organise a replacement.\n\n Customer complaint: Dear Customer Service,I recently purchased an iPad, but I am quite disappointed with the product. I\nhave tried multiple times to connect it to my computer, but the process is\nextremely difficult and frustrating. Additionally, the battery life is\nsignificantly shorter than advertised. I hope you can find a solution to this\nissue soon. Regards, John Smith\n Customer request: an iPad\n Response:\n","\n Dear John Smith,\n\n Please wait for the customer service representative to reply.\n\nA replacement iPad is the best way to ensure your iPad is working properly. Many people are\n\nnot happy about the problems with the iPad's battery life. We are currently working to resolve\n\nthe issue.\n\nA replacement iPad is the best way to ensure your iPad is working properly. Many people are\n\nnot happy about the problems with the iPad's battery life. We are currently working to resolve the\n\nproblem. We will send you an application to resolve this issue.\n\nYou will be d..."
2,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt,Sarah Johnson,Your customer service is\nunresponsive,"\n Customer complaint : My order is faulty.\n Customer request: A refund\n Response: We apologise for this and we will organise a replacement.\n\n Customer complaint: Dear Customer Service,I am writing to express my extreme disappointment with the iPad I received. The\ndevice has been functioning poorly since the day I bought it, and I have already\nhad to return it twice due to technical issues. Your customer service is\nunresponsive and does not seem to care about resolving my problems. I will no\nlonger be purchasing from your company in the future. Regards, Sarah Johnson\n ...","\n Dear Sarah Johnson,\n\n Reply: \n\nSarah Johnson\n\nCustomer Service\n\nDear Customer Service,\n\nI received the iPad after I ordered from the store. My computer was so bad that it started to crack. It was the only thing that kept me from returning it. The only thing I could do was to open the computer and try to run a small test of the iPad. But before I could do that, I received a call from the store, and my computer had started to crack. I immediately ordered the replacement, and am very happy with it. I received the replacement as quickly as I could.\n\n..."
3,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt,Michael Brown,iPad,"\n Customer complaint : My order is faulty.\n Customer request: A refund\n Response: We apologise for this and we will organise a replacement.\n\n 

# Reflection
1. It was an interesting exercise to go through.  I probably focussed too much on the text generation side of things, but it was interesting to see what could be achieved.
1. I can certainly see how you could get quite frustrated by getting close to a solution, only to see that the results may not be reliably achievable.

# References
1. https://towardsdatascience.com/choosing-and-implementing-hugging-face-models-026d71426fbe/
1. https://huggingface.co/blog/MichielBontenbal/how-to-choose-your-ai-model
1. https://bentoml.com/llm/getting-started/choosing-the-right-model

# Workings


In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

In [ ]:
#generated_text, prompt_used = generate_letter_response(customer_feedback_records[0])
#
#
#print(generated_text)
#print ('-' * 50)
#response_text = generated_text; #[len(prompt):]
#
#response_formatted = textwrap.fill(response_text, width=80)
#print(response_formatted)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dear Bumblebee,

I sincerely apologize for the confusion and inconvenience you've experienced with your recent order. We deeply regret sending you the wrong figure. 

To rectify this, we will promptly exchange your Megatron action figure for the Optimus Prime you originally ordered. Please return the Megatron as directed and we'll process the exchange.

Thank you for bringing this to our attention. We appreciate your patience and understanding.

Best regards,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I'm truly sorry to hear about the mix-up with your Optimus Prime action figure. We regretfully sent you the Megatron instead. 

To resolve this, we will exchange your Megatron for the Optimus Prime you ordered. Please return the Megatron and we'll process the exchange promptly.

Thank you for your patience and understanding.

Best,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I sincerely apologize for the mix-up with your Optimus Prime actio

In [ ]:
#summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
#
#outputs = summarizer(text, max_length=130, min_length=30, do_sample=False)
#outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
#print(outputs[0]['summary_text'])

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
def test_ner_model(ner_pipeline):

    for feedback in customer_feedback_records:

        print (f"Url {feedback.url}")

        test_text  = feedback.original_letter

        feedback.customer_name = extract_customer_name(ner_pipeline, test_text)

        ner_results= extract_customer_name(ner_pipeline, test_text)

        pprint(ner_results)
